In [52]:
# Importing necessary libraries
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from typing import (
    Literal,
)

import torch
import torch.nn.functional as F # Padding
from torch.utils.data import Dataset, DataLoader
from torch.nn import Module, Embedding, RNN, Linear

# Device setup
device: Literal['cpu', 'cuda'] = 'cuda' if torch.cuda.is_available() else 'cpu'

In [53]:
# Loading the dataset
df: pd.DataFrame = pd.read_csv(filepath_or_buffer="data/questions and answers.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [54]:
# Preprocessing
def preprocess(text: str) -> list:
    text = text.lower()
    text = text.replace("?", "").replace("'", "")
    return text

# Vocabulary building
vocab: dict[str, int] = {'<UNK>': 0}

def build_vocab(row: pd.Series) -> None:
    merged_tokens: list[str] = preprocess(row['question']) + preprocess(row['answer'])
    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

df.apply(build_vocab, axis=1)
print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 41


In [55]:
# Dataset
class CustomDataset(Dataset):
    def __init__(self, df, vocab):
        super().__init__()
        self.df = df
        self.vocab = vocab

        self.vectorizer = CountVectorizer(
            max_features=200,
            preprocessor=preprocess,
            binary=False, # Counting and not Presence
        )
        self.vectorizer.fit(raw_documents=df['question'].tolist() + df['answer'].tolist())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index: int):
        doc = self.df.iloc[index]
        question = self.vectorizer.transform([doc['question']]).toarray()
        answer = self.vectorizer.transform([doc['answer']]).toarray()
        return torch.tensor(question, dtype=torch.float32), torch.tensor(answer, dtype=torch.float32)

In [57]:
# Building Dataloaders
dataset = CustomDataset(df=df, vocab=vocab)
dataloader: DataLoader = DataLoader(dataset=dataset, batch_size=5, shuffle=True)